# QELM sur la matrice de mesures M, balayage en p

Expérience `experiments/QELM_M/exp_QELM_M.py` (issue #94). Pour chaque échantillon, le réservoir (TFIM désordonné, ancilla intriquée avec la charge du système) évolue pendant `evolution.t`, découpé en `Ns` pas. Après chaque pas, chaque qubit du système est mesuré en Z avec la probabilité `p`. Les features sont le relevé des mesures `M` (0 pour un qubit non mesuré), et la cible est `<Z_0>` de l'ancilla, jamais mesurée, à l'instant final.

Ce notebook lance le balayage avec `experiments/QELM_M/exp_QELM_M.sh`, relit les dossiers `outputs/exp_QELM_M/<date_heure>/N<N>_seed<seed>_p<p>/`, puis trace R2 et NMSE en fonction de `p`, moyennés sur les seeds avec l'erreur standard en barres d'erreur.

À p = 1, mesurer tout le système fixe la charge, donc `<Z_0>` vaut ±1 et se lit sur la dernière ligne de M : R2 ≈ 1 y est attendu quelle que soit la dynamique (issue #104).

In [ ]:
import os
from pathlib import Path

# Jupyter starts in notebooks/. Move to the repository root (the folder that
# contains pyproject.toml) so experiments/ and outputs/ resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Paramètres

Chaque valeur peut être changée ici, ou par la variable d'environnement du même nom (c'est ce que fait le test `tests/test_qelm_m_notebook.py` pour un petit balayage). `RUN = False` relit le dernier balayage sans en relancer un.

In [ ]:
L = os.environ.get("L", "500")  # samples per run
N_VALUES = os.environ.get("N_VALUES", "4")  # total qubits, ancilla included
SEEDS = os.environ.get("SEEDS", "42 43 44")
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.5 0.8 1.0")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Lancer le balayage

In [ ]:
from QELM.utils import latest_sweep, run_with_progress

if RUN:
    env = {
        **os.environ,
        "L": L,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "OUT_ROOT": str(OUT_ROOT),
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(P_VALUES.split())
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/QELM_M/exp_QELM_M.sh"], total=n_runs, env=env
    )

sweep_dir = latest_sweep(OUT_ROOT / "exp_QELM_M")
print(f"sweep: {sweep_dir}")

## Relire les métriques

In [ ]:
from QELM.utils import load_sweep, summarize

rows = load_sweep(sweep_dir)
table = summarize(rows, by=("N", "p"))
print(f"{len(rows)} runs")
print(f"{'N':>3} {'p':>5} {'runs':>4} {'R2':>16} {'NMSE':>16}")
for r in table:
    print(
        f"{r['N']:>3} {r['p']:>5.2f} {r['n_runs']:>4} "
        f"{r['r2_mean']:>8.4f} ± {r['r2_sem']:<6.4f} "
        f"{r['nmse_mean']:>8.4f} ± {r['nmse_sem']:<6.4f}"
    )

## R2 et NMSE en fonction de p

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax_r2, ax_nmse) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for N in sorted({r["N"] for r in table}):
    sub = [r for r in table if r["N"] == N]
    p = [r["p"] for r in sub]
    # One seed gives no spread: draw the points without error bars.
    r2_err = np.nan_to_num([r["r2_sem"] for r in sub])
    nmse_err = np.nan_to_num([r["nmse_sem"] for r in sub])
    ax_r2.errorbar(
        p,
        [r["r2_mean"] for r in sub],
        yerr=r2_err,
        marker="o",
        capsize=3,
        label=f"N = {N}",
    )
    ax_nmse.errorbar(
        p,
        [r["nmse_mean"] for r in sub],
        yerr=nmse_err,
        marker="o",
        capsize=3,
        label=f"N = {N}",
    )
ax_r2.set(xlabel="measurement probability p", ylabel="R2 (test)", title="R2 vs p")
ax_nmse.set(
    xlabel="measurement probability p",
    ylabel="NMSE (test)",
    title="NMSE vs p",
    yscale="log",
)
for ax in (ax_r2, ax_nmse):
    ax.grid(alpha=0.3)
    ax.legend()
figure_path = sweep_dir / "metrics_vs_p.png"
fig.savefig(figure_path, dpi=150)
print(f"figure: {figure_path}")
plt.show()